In [15]:
"""
===============================================================================
Project Title:
Machine Learning-Based Early Risk Prediction for Common Diseases

File:
train_model.py

Author:
Ashu-Nkhangu Fidelis

Department:
Software Engineering

Description
-----------
This script trains four independent machine learning models for early disease
risk prediction using Random Forest Classifiers.

Diseases Supported
------------------
1. Diabetes
2. Heart Disease
3. Chronic Kidney Disease
4. Stroke

Major Responsibilities
----------------------
✓ Load datasets
✓ Validate datasets
✓ Clean datasets
✓ Remove duplicate records
✓ Handle missing values
✓ Remove medically impossible values
✓ Encode categorical variables
✓ Scale numerical variables
✓ Generate exploratory visualizations
✓ Train Random Forest models
✓ Evaluate performance
✓ Save trained models
✓ Save evaluation reports
✓ Save figures for Chapter Four of the project report

===============================================================================
"""

# =============================================================================
# IMPORTS
# =============================================================================

import os
import warnings
import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import seaborn as sns

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.impute import (
    SimpleImputer
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.ensemble import (
    RandomForestClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve
)

# =============================================================================
# WARNING CONFIGURATION
# =============================================================================

warnings.filterwarnings("ignore")

# =============================================================================
# MATPLOTLIB SETTINGS
# =============================================================================

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

sns.set_style("whitegrid")

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================

BASE_DIR = Path.cwd().parent

DATASET_DIR = BASE_DIR / "datasets"

MODEL_DIR = BASE_DIR / "backend" / "app" / "models"

REPORT_DIR = BASE_DIR / "reports"

FIGURE_DIR = BASE_DIR / "figures"

# =============================================================================
# AUTOMATIC DIRECTORY CREATION
# =============================================================================

# MODEL_DIR.mkdir(exist_ok=True)

REPORT_DIR.mkdir(exist_ok=True)

FIGURE_DIR.mkdir(exist_ok=True)

# =============================================================================
# FIGURE SUBDIRECTORIES
# =============================================================================

(FIGURE_DIR / "heart").mkdir(exist_ok=True)

(FIGURE_DIR / "diabetes").mkdir(exist_ok=True)

(FIGURE_DIR / "kidney").mkdir(exist_ok=True)

(FIGURE_DIR / "stroke").mkdir(exist_ok=True)

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

LOGGER = logging.getLogger(__name__)

# =============================================================================
# DATASET CONFIGURATION
# =============================================================================

DATASETS = {

    "Heart": {

        "File": DATASET_DIR / "heart.csv",

        "Target": "TargetBinary",

        "Columns": [

            "Age",
            "Sex",
            "Cp",
            "Trestbps",
            "Chol",
            "Fbs",
            "Restecg",
            "Thalach",
            "Exang",
            "Oldpeak",
            "Slope",
            "Ca",
            "Thal",
            "Num",
            "TargetBinary"

        ],

        "CriticalColumns": [

            "Age",
            "Sex",
            "Cp",
            "TargetBinary"

        ],

        "ZeroInvalidColumns": [

            "Trestbps",
            "Chol",
            "Thalach"

        ]

    },

    "Diabetes": {

        "File": DATASET_DIR / "diabetes.csv",

        "Target": "Outcome",

        "Columns": [

            "Pregnancies",
            "Glucose",
            "BloodPressure",
            "SkinThickness",
            "Insulin",
            "BMI",
            "DiabetesPedigreeFunction",
            "Age",
            "Outcome"

        ],

        "CriticalColumns": [

            "Glucose",
            "BMI",
            "Age",
            "Outcome"

        ],

        "ZeroInvalidColumns": [

            "Glucose",
            "BloodPressure",
            "SkinThickness",
            "Insulin",
            "BMI"

        ]

    },

    "Kidney": {

        "File": DATASET_DIR / "kidney.csv",

        "Target": "Class",

        "Columns": [

            "Bp",
            "Sg",
            "Al",
            "Su",
            "Rbc",
            "Bu",
            "Sc",
            "Sod",
            "Pot",
            "Hemo",
            "Wbcc",
            "Rbcc",
            "Htn",
            "Class"

        ],

        "CriticalColumns": [

            "Bp",
            "Hemo",
            "Class"

        ],

        "ZeroInvalidColumns": [

            "Bp",
            "Bu",
            "Sc",
            "Hemo"

        ]

    },

    "Stroke": {

        "File": DATASET_DIR / "stroke.csv",

        "Target": "Stroke",

        "Columns": [

            "Id",
            "Gender",
            "Age",
            "Hypertension",
            "HeartDisease",
            "EverMarried",
            "WorkType",
            "ResidenceType",
            "AvgGlucoseLevel",
            "Bmi",
            "SmokingStatus",
            "Stroke"

        ],

        "CriticalColumns": [

            "Age",
            "Gender",
            "Stroke"

        ],

        "ZeroInvalidColumns": [

            "Age",
            "AvgGlucoseLevel",
            "Bmi"

        ]

    }

}

# =============================================================================
# RANDOM STATE
# =============================================================================

RANDOM_STATE = 42

TEST_SIZE = 0.20

N_ESTIMATORS = 300

# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def print_separator(length: int = 80) -> None:
    """
    Prints a separator line in the console.

    Parameters
    ----------
    length : int
        Length of separator.
    """
    print("=" * length)


def print_title(title: str) -> None:
    """
    Prints a formatted title.

    Parameters
    ----------
    title : str
        Title to display.
    """
    print_separator()
    print(title)
    print_separator()


def save_current_figure(path: Path) -> None:
    """
    Saves the currently active matplotlib figure.

    Parameters
    ----------
    path : Path
        Destination file path.
    """

    plt.tight_layout()
    plt.savefig(path)
    plt.close()


def log_dataset_shape(
    dataframe: pd.DataFrame,
    dataset_name: str
) -> None:
    """
    Logs dataset dimensions.

    Parameters
    ----------
    dataframe : DataFrame

    dataset_name : str
    """

    LOGGER.info(
        "%s Dataset Shape: %s",
        dataset_name,
        dataframe.shape
    )


def ensure_columns_exist(
    dataframe: pd.DataFrame,
    expected_columns: list
) -> None:
    """
    Validates required columns.

    Raises
    ------
    ValueError
        If any required column is missing.
    """

    missing = [
        column
        for column in expected_columns
        if column not in dataframe.columns
    ]

    if missing:

        raise ValueError(
            f"Missing required columns: {missing}"
        )

In [16]:
# =============================================================================
# DATA LOADING FUNCTIONS
# =============================================================================

def load_dataset(
    dataset_name: str,
    dataset_config: dict
) -> pd.DataFrame:
    """
    Loads a dataset from disk and performs initial validation.

    Parameters
    ----------
    dataset_name : str
        Name of the dataset.

    dataset_config : dict
        Dataset configuration dictionary.

    Returns
    -------
    pd.DataFrame
        Loaded dataframe.
    """

    LOGGER.info("Loading %s dataset...", dataset_name)

    dataset_path = dataset_config["File"]

    if not dataset_path.exists():

        raise FileNotFoundError(
            f"Dataset not found: {dataset_path}"
        )

    dataframe = pd.read_csv(dataset_path)

    log_dataset_shape(dataframe, dataset_name)

    ensure_columns_exist(
        dataframe,
        dataset_config["Columns"]
    )

    LOGGER.info(
        "%s dataset loaded successfully.",
        dataset_name
    )

    return dataframe


# =============================================================================
# DATA CLEANING FUNCTIONS
# =============================================================================

def remove_duplicate_rows(
    dataframe: pd.DataFrame,
    dataset_name: str
) -> pd.DataFrame:
    """
    Removes duplicate observations.

    Parameters
    ----------
    dataframe : pd.DataFrame

    dataset_name : str

    Returns
    -------
    pd.DataFrame
    """

    before = len(dataframe)

    dataframe = dataframe.drop_duplicates()

    after = len(dataframe)

    LOGGER.info(
        "%s | Removed %d duplicate rows.",
        dataset_name,
        before - after
    )

    return dataframe


def remove_rows_with_missing_targets(
    dataframe: pd.DataFrame,
    target_column: str
) -> pd.DataFrame:
    """
    Removes observations with missing target labels.

    Parameters
    ----------
    dataframe : pd.DataFrame

    target_column : str

    Returns
    -------
    pd.DataFrame
    """

    dataframe = dataframe.dropna(
        subset=[target_column]
    )

    return dataframe


def remove_rows_with_missing_critical_columns(
    dataframe: pd.DataFrame,
    critical_columns: list,
    dataset_name: str
) -> pd.DataFrame:
    """
    Removes rows where essential variables are missing.

    Parameters
    ----------
    dataframe : pd.DataFrame

    critical_columns : list

    dataset_name : str
    """

    before = len(dataframe)

    dataframe = dataframe.dropna(
        subset=critical_columns
    )

    after = len(dataframe)

    LOGGER.info(
        "%s | Removed %d rows with critical missing values.",
        dataset_name,
        before - after
    )

    return dataframe


# =============================================================================
# MEDICALLY IMPOSSIBLE VALUE HANDLING
# =============================================================================

def replace_invalid_zero_values(
    dataframe: pd.DataFrame,
    columns: list,
    dataset_name: str
) -> pd.DataFrame:
    """
    Replaces medically impossible zero values with NaN.

    The affected values will later be imputed.

    Parameters
    ----------
    dataframe : DataFrame

    columns : list

    dataset_name : str

    Returns
    -------
    DataFrame
    """

    for column in columns:

        if column not in dataframe.columns:
            continue

        count = (dataframe[column] == 0).sum()

        if count > 0:

            LOGGER.info(
                "%s | %d invalid zero values detected in %s.",
                dataset_name,
                count,
                column
            )

            dataframe[column] = dataframe[column].replace(
                0,
                np.nan
            )

    return dataframe


# =============================================================================
# DATA TYPE STANDARDIZATION
# =============================================================================

def convert_numeric_columns(
    dataframe: pd.DataFrame
) -> pd.DataFrame:
    """
    Attempts to convert every column to numeric whenever possible.

    Non-convertible columns remain unchanged.
    """

    for column in dataframe.columns:

        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="ignore"
        )

    return dataframe


def strip_whitespace_from_strings(
    dataframe: pd.DataFrame
) -> pd.DataFrame:
    """
    Removes leading and trailing whitespace from
    categorical columns.
    """

    object_columns = dataframe.select_dtypes(
        include=["object"]
    ).columns

    for column in object_columns:

        dataframe[column] = dataframe[column].astype(str)

        dataframe[column] = dataframe[column].str.strip()

    return dataframe


# =============================================================================
# FEATURE IDENTIFICATION
# =============================================================================

def identify_feature_types(
    dataframe: pd.DataFrame,
    target_column: str
):
    """
    Determines numerical and categorical predictors.

    Returns
    -------
    tuple
        numerical_columns,
        categorical_columns
    """

    features = dataframe.drop(
        columns=[target_column]
    )

    numerical_columns = features.select_dtypes(
        include=[
            np.number
        ]
    ).columns.tolist()

    categorical_columns = features.select_dtypes(
        exclude=[
            np.number
        ]
    ).columns.tolist()

    LOGGER.info(
        "Detected %d numerical features.",
        len(numerical_columns)
    )

    LOGGER.info(
        "Detected %d categorical features.",
        len(categorical_columns)
    )

    return (
        numerical_columns,
        categorical_columns
    )


# =============================================================================
# MISSING VALUE IMPUTATION
# =============================================================================

def create_preprocessing_pipeline(
    numerical_columns: list,
    categorical_columns: list
) -> ColumnTransformer:
    """
    Creates a reusable preprocessing pipeline.

    Numerical Features
    ------------------
    • Median Imputation
    • Standard Scaling

    Categorical Features
    --------------------
    • Most Frequent Imputation
    • One-Hot Encoding
    """

    numeric_pipeline = Pipeline(

        steps=[

            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            )

        ]

    )

    categorical_pipeline = Pipeline(

        steps=[

            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),

            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )

        ]

    )

    transformer = ColumnTransformer(

        transformers=[

            (
                "Numeric",
                numeric_pipeline,
                numerical_columns
            ),

            (
                "Categorical",
                categorical_pipeline,
                categorical_columns
            )

        ]

    )

    return transformer


# =============================================================================
# TARGET ENCODING
# =============================================================================

def encode_target_if_needed(
    dataframe: pd.DataFrame,
    target_column: str
):
    """
    Encodes target labels whenever they are categorical.

    Returns
    -------
    dataframe,
    label_encoder
    """

    encoder = None

    if dataframe[target_column].dtype == object:

        encoder = LabelEncoder()

        dataframe[target_column] = encoder.fit_transform(
            dataframe[target_column]
        )

        LOGGER.info(
            "Encoded target labels."
        )

    return dataframe, encoder


# =============================================================================
# COMPLETE DATA PREPARATION
# =============================================================================

def prepare_dataset(
    dataset_name: str,
    dataset_config: dict
):
    """
    Performs the complete preprocessing workflow before
    model training.

    Workflow
    --------
    1. Load dataset
    2. Remove duplicates
    3. Remove missing targets
    4. Remove missing critical rows
    5. Replace impossible zero values
    6. Standardize data types
    7. Encode target if necessary
    8. Detect feature types
    9. Create preprocessing pipeline

    Returns
    -------
    tuple
        dataframe,
        preprocessing_pipeline,
        label_encoder,
        numeric_columns,
        categorical_columns
    """

    dataframe = load_dataset(
        dataset_name,
        dataset_config
    )

    dataframe = remove_duplicate_rows(
        dataframe,
        dataset_name
    )

    dataframe = remove_rows_with_missing_targets(
        dataframe,
        dataset_config["Target"]
    )

    dataframe = remove_rows_with_missing_critical_columns(
        dataframe,
        dataset_config["CriticalColumns"],
        dataset_name
    )

    dataframe = replace_invalid_zero_values(
        dataframe,
        dataset_config["ZeroInvalidColumns"],
        dataset_name
    )

    dataframe = strip_whitespace_from_strings(
        dataframe
    )

    dataframe = convert_numeric_columns(
        dataframe
    )

    dataframe, label_encoder = encode_target_if_needed(
        dataframe,
        dataset_config["Target"]
    )

    (
        numeric_columns,
        categorical_columns
    ) = identify_feature_types(
        dataframe,
        dataset_config["Target"]
    )

    preprocessing_pipeline = create_preprocessing_pipeline(
        numeric_columns,
        categorical_columns
    )

    LOGGER.info(
        "%s preprocessing complete.",
        dataset_name
    )

    return (
        dataframe,
        preprocessing_pipeline,
        label_encoder,
        numeric_columns,
        categorical_columns
    )

In [17]:
# =============================================================================
# DATASET SPLITTING FUNCTIONS
# =============================================================================

def split_features_and_target(
    dataframe: pd.DataFrame,
    target_column: str
):
    """
    Separates predictor variables from the target variable.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Input dataset.

    target_column : str
        Name of the target column.

    Returns
    -------
    tuple
        X, y
    """

    X = dataframe.drop(columns=[target_column])

    y = dataframe[target_column]

    LOGGER.info(
        "Feature matrix shape: %s | Target shape: %s",
        X.shape,
        y.shape
    )

    return X, y


def perform_train_test_split(
    X,
    y
):
    """
    Splits the dataset into training and testing sets.

    A stratified split is used to preserve the original
    class distribution whenever possible.

    Parameters
    ----------
    X : pd.DataFrame

    y : pd.Series

    Returns
    -------
    tuple
        XTrain,
        XTest,
        yTrain,
        yTest
    """

    XTrain, XTest, yTrain, yTest = train_test_split(

        X,

        y,

        test_size=TEST_SIZE,

        random_state=RANDOM_STATE,

        stratify=y

    )

    LOGGER.info(
        "Training samples : %d",
        len(XTrain)
    )

    LOGGER.info(
        "Testing samples  : %d",
        len(XTest)
    )

    return (

        XTrain,

        XTest,

        yTrain,

        yTest

    )


# =============================================================================
# PREPROCESSING TRANSFORMATION
# =============================================================================

def preprocess_training_data(
    preprocessing_pipeline: ColumnTransformer,
    XTrain: pd.DataFrame,
    XTest: pd.DataFrame
):
    """
    Fits the preprocessing pipeline using only the
    training dataset.

    Parameters
    ----------
    preprocessing_pipeline : ColumnTransformer

    XTrain : DataFrame

    XTest : DataFrame

    Returns
    -------
    tuple
        XTrainProcessed,
        XTestProcessed
    """

    LOGGER.info(
        "Fitting preprocessing pipeline..."
    )

    XTrainProcessed = preprocessing_pipeline.fit_transform(
        XTrain
    )

    XTestProcessed = preprocessing_pipeline.transform(
        XTest
    )

    LOGGER.info(
        "Preprocessing complete."
    )

    LOGGER.info(
        "Processed training shape: %s",
        XTrainProcessed.shape
    )

    LOGGER.info(
        "Processed testing shape: %s",
        XTestProcessed.shape
    )

    return (

        XTrainProcessed,

        XTestProcessed

    )


# =============================================================================
# RANDOM FOREST MODEL CREATION
# =============================================================================

def create_random_forest_model():
    """
    Creates the Random Forest classifier used
    throughout the project.

    Returns
    -------
    RandomForestClassifier
    """

    model = RandomForestClassifier(

        n_estimators=N_ESTIMATORS,

        criterion="gini",

        max_depth=None,

        min_samples_split=2,

        min_samples_leaf=1,

        bootstrap=True,

        random_state=RANDOM_STATE,

        n_jobs=-1

    )

    return model


# =============================================================================
# MODEL TRAINING
# =============================================================================

def train_model(
    model,
    XTrain,
    yTrain,
    dataset_name: str
):
    """
    Trains the supplied machine learning model.

    Parameters
    ----------
    model

    XTrain

    yTrain

    dataset_name : str
    """

    LOGGER.info(
        "Training %s Random Forest model...",
        dataset_name
    )

    model.fit(

        XTrain,

        yTrain

    )

    LOGGER.info(
        "%s training completed successfully.",
        dataset_name
    )

    return model


# =============================================================================
# MODEL PREDICTION
# =============================================================================

def generate_predictions(
    model,
    XTest
):
    """
    Generates predictions and prediction probabilities.

    Parameters
    ----------
    model

    XTest

    Returns
    -------
    tuple
        Predictions,
        Probabilities
    """

    predictions = model.predict(

        XTest

    )

    probabilities = None

    if hasattr(
        model,
        "predict_proba"
    ):

        probabilities = model.predict_proba(
            XTest
        )[:, 1]

    return (

        predictions,

        probabilities

    )


# =============================================================================
# MODEL EVALUATION
# =============================================================================

def evaluate_model(
    yTest,
    predictions
):
    """
    Computes the primary classification metrics.

    Parameters
    ----------
    yTest

    predictions

    Returns
    -------
    dict
    """

    metrics = {

        "Accuracy": accuracy_score(

            yTest,

            predictions

        ),

        "Precision": precision_score(

            yTest,

            predictions,

            zero_division=0

        ),

        "Recall": recall_score(

            yTest,

            predictions,

            zero_division=0

        ),

        "F1Score": f1_score(

            yTest,

            predictions,

            zero_division=0

        )

    }

    LOGGER.info(

        "Accuracy : %.4f",

        metrics["Accuracy"]

    )

    LOGGER.info(

        "Precision : %.4f",

        metrics["Precision"]

    )

    LOGGER.info(

        "Recall : %.4f",

        metrics["Recall"]

    )

    LOGGER.info(

        "F1 Score : %.4f",

        metrics["F1Score"]

    )

    return metrics


# =============================================================================
# CLASSIFICATION REPORT
# =============================================================================

def generate_classification_report(
    yTest,
    predictions
):
    """
    Produces the complete classification report.

    Returns
    -------
    str
    """

    report = classification_report(

        yTest,

        predictions,

        digits=4

    )

    return report


# =============================================================================
# CONFUSION MATRIX DATA
# =============================================================================

def compute_confusion_matrix(
    yTest,
    predictions
):
    """
    Computes the confusion matrix.

    Returns
    -------
    ndarray
    """

    matrix = confusion_matrix(

        yTest,

        predictions

    )

    return matrix


# =============================================================================
# FEATURE NAME EXTRACTION
# =============================================================================

def get_feature_names(
    preprocessing_pipeline,
    numerical_columns,
    categorical_columns
):
    """
    Retrieves transformed feature names after
    preprocessing.

    Returns
    -------
    list
    """

    feature_names = []

    feature_names.extend(
        numerical_columns
    )

    if len(categorical_columns) > 0:

        encoder = preprocessing_pipeline.named_transformers_[

            "Categorical"

        ].named_steps["encoder"]

        encoded_columns = encoder.get_feature_names_out(

            categorical_columns

        )

        feature_names.extend(

            encoded_columns.tolist()

        )

    return feature_names


# =============================================================================
# MODEL SERIALIZATION
# =============================================================================

def save_model(
    model,
    preprocessing_pipeline,
    label_encoder,
    dataset_name: str
):
    """
    Saves the trained model together with the
    preprocessing pipeline.

    The saved object includes:

    • Model
    • Pipeline
    • Label Encoder

    Returns
    -------
    Path
    """

    output = {

        "Model": model,

        "Preprocessor": preprocessing_pipeline,

        "LabelEncoder": label_encoder

    }

    model_path = MODEL_DIR / f"{dataset_name.lower()}_model.pkl"

    joblib.dump(

        output,

        model_path

    )

    LOGGER.info(

        "%s model saved to %s",

        dataset_name,

        model_path

    )

    return model_path

In [18]:
# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================

def plot_missing_values(
    dataframe: pd.DataFrame,
    dataset_name: str
) -> None:
    """
    Generates a bar chart showing the number of missing values
    for each feature before preprocessing.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Dataset to analyze.

    dataset_name : str
        Dataset identifier.
    """

    missing_counts = dataframe.isnull().sum()

    missing_counts = missing_counts[
        missing_counts > 0
    ].sort_values(ascending=False)

    if missing_counts.empty:

        LOGGER.info(
            "%s | No missing values detected.",
            dataset_name
        )

        return

    plt.figure(figsize=(10, 6))

    missing_counts.plot(
        kind="bar"
    )

    plt.title(
        f"{dataset_name} Missing Values"
    )

    plt.xlabel("Features")

    plt.ylabel("Missing Values")

    plt.xticks(rotation=45)

    save_current_figure(
        FIGURE_DIR /
        dataset_name.lower() /
        "missing_values.png"
    )


# =============================================================================
# CORRELATION HEATMAP
# =============================================================================

def plot_correlation_heatmap(
    dataframe: pd.DataFrame,
    dataset_name: str
) -> None:
    """
    Generates a Pearson correlation heatmap
    using only numerical variables.
    """

    numeric_df = dataframe.select_dtypes(
        include=np.number
    )

    if numeric_df.shape[1] < 2:

        LOGGER.info(
            "%s | Correlation heatmap skipped.",
            dataset_name
        )

        return

    plt.figure(figsize=(12, 10))

    sns.heatmap(

        numeric_df.corr(),

        annot=True,

        fmt=".2f",

        cmap="coolwarm",

        square=True

    )

    plt.title(
        f"{dataset_name} Correlation Heatmap"
    )

    save_current_figure(

        FIGURE_DIR /
        dataset_name.lower() /
        "correlation_heatmap.png"

    )


# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

def plot_class_distribution(
    dataframe: pd.DataFrame,
    target_column: str,
    dataset_name: str
) -> None:
    """
    Visualizes target class frequencies.
    """

    plt.figure(figsize=(7, 5))

    sns.countplot(

        x=target_column,

        data=dataframe

    )

    plt.title(
        f"{dataset_name} Class Distribution"
    )

    plt.ylabel("Frequency")

    save_current_figure(

        FIGURE_DIR /
        dataset_name.lower() /
        "class_distribution.png"

    )


# =============================================================================
# FEATURE DISTRIBUTIONS
# =============================================================================

def plot_feature_distributions(
    dataframe: pd.DataFrame,
    dataset_name: str
) -> None:
    """
    Creates histogram plots for every
    numerical feature.
    """

    numeric_columns = dataframe.select_dtypes(

        include=np.number

    ).columns

    for column in numeric_columns:

        plt.figure(figsize=(8, 5))

        sns.histplot(

            dataframe[column],

            kde=True,

            bins=30

        )

        plt.title(

            f"{dataset_name} - {column}"

        )

        plt.xlabel(column)

        plt.ylabel("Frequency")

        filename = (
            f"distribution_{column.lower()}.png"
        )

        save_current_figure(

            FIGURE_DIR /
            dataset_name.lower() /
            filename

        )


# =============================================================================
# BOXPLOTS
# =============================================================================

def plot_boxplots(
    dataframe: pd.DataFrame,
    dataset_name: str
) -> None:
    """
    Generates boxplots for every numerical feature
    to assist with outlier detection.
    """

    numeric_columns = dataframe.select_dtypes(

        include=np.number

    ).columns

    for column in numeric_columns:

        plt.figure(figsize=(8, 4))

        sns.boxplot(

            x=dataframe[column]

        )

        plt.title(

            f"{dataset_name} - {column}"

        )

        filename = (
            f"boxplot_{column.lower()}.png"
        )

        save_current_figure(

            FIGURE_DIR /
            dataset_name.lower() /
            filename

        )


# =============================================================================
# CONFUSION MATRIX
# =============================================================================

def plot_confusion_matrix(
    matrix,
    dataset_name: str
) -> None:
    """
    Saves the confusion matrix figure.
    """

    plt.figure(figsize=(6, 5))

    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues"

    )

    plt.title(
        f"{dataset_name} Confusion Matrix"
    )

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    save_current_figure(

        FIGURE_DIR /
        dataset_name.lower() /
        "confusion_matrix.png"

    )


# =============================================================================
# ROC CURVE
# =============================================================================

def plot_roc_curve(
    y_test,
    probabilities,
    dataset_name: str
) -> None:
    """
    Generates ROC Curve.
    """

    if probabilities is None:

        LOGGER.info(
            "%s | ROC skipped.",
            dataset_name
        )

        return

    fpr, tpr, _ = roc_curve(

        y_test,

        probabilities

    )

    roc_auc = auc(

        fpr,

        tpr

    )

    plt.figure(figsize=(7, 6))

    plt.plot(

        fpr,

        tpr,

        linewidth=2,

        label=f"AUC = {roc_auc:.3f}"

    )

    plt.plot(

        [0, 1],

        [0, 1],

        linestyle="--"

    )

    plt.xlabel("False Positive Rate")

    plt.ylabel("True Positive Rate")

    plt.title(

        f"{dataset_name} ROC Curve"

    )

    plt.legend()

    save_current_figure(

        FIGURE_DIR /
        dataset_name.lower() /
        "roc_curve.png"

    )


# =============================================================================
# PRECISION-RECALL CURVE
# =============================================================================

def plot_precision_recall_curve(
    y_test,
    probabilities,
    dataset_name: str
) -> None:
    """
    Generates Precision-Recall Curve.
    """

    if probabilities is None:

        LOGGER.info(
            "%s | PR Curve skipped.",
            dataset_name
        )

        return

    precision, recall, _ = precision_recall_curve(

        y_test,

        probabilities

    )

    plt.figure(figsize=(7, 6))

    plt.plot(

        recall,

        precision,

        linewidth=2

    )

    plt.xlabel("Recall")

    plt.ylabel("Precision")

    plt.title(

        f"{dataset_name} Precision-Recall Curve"

    )

    save_current_figure(

        FIGURE_DIR /
        dataset_name.lower() /
        "precision_recall_curve.png"

    )


# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================

def plot_feature_importance(
    model,
    feature_names,
    dataset_name: str,
    top_n: int = 20
) -> None:
    """
    Visualizes the most important features
    identified by the Random Forest model.
    """

    importance = pd.DataFrame({

        "Feature": feature_names,

        "Importance": model.feature_importances_

    })

    importance = importance.sort_values(

        by="Importance",

        ascending=False

    ).head(top_n)

    plt.figure(figsize=(10, 7))

    sns.barplot(

        data=importance,

        x="Importance",

        y="Feature"

    )

    plt.title(

        f"{dataset_name} Feature Importance"

    )

    save_current_figure(

        FIGURE_DIR /
        dataset_name.lower() /
        "feature_importance.png"

    )


# =============================================================================
# ACCURACY COMPARISON
# =============================================================================

def plot_accuracy_comparison(
    accuracy_dictionary: dict
) -> None:
    """
    Generates a comparison plot showing
    the accuracy of all trained disease models.
    """

    plt.figure(figsize=(8, 6))

    diseases = list(

        accuracy_dictionary.keys()

    )

    accuracies = list(

        accuracy_dictionary.values()

    )

    plt.bar(

        diseases,

        accuracies

    )

    plt.gca().yaxis.set_major_formatter(

        ticker.PercentFormatter(xmax=1)

    )

    plt.ylim(0, 1)

    plt.ylabel("Accuracy")

    plt.title(

        "Accuracy Comparison Across Disease Models"

    )

    save_current_figure(

        FIGURE_DIR /
        "accuracy_comparison.png"

    )

In [19]:
# =============================================================================
# REPORT SAVING FUNCTIONS
# =============================================================================

def save_classification_report(
    report: str,
    dataset_name: str
) -> Path:
    """
    Saves the classification report to the reports directory.

    Parameters
    ----------
    report : str
        Classification report generated by scikit-learn.

    dataset_name : str
        Dataset name.

    Returns
    -------
    Path
        Saved report path.
    """

    report_path = (
        REPORT_DIR /
        f"{dataset_name.lower()}_classification_report.txt"
    )

    with open(
        report_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(
            "=" * 80 + "\n"
        )

        file.write(
            f"{dataset_name.upper()} CLASSIFICATION REPORT\n"
        )

        file.write(
            "=" * 80 + "\n\n"
        )

        file.write(report)

    LOGGER.info(
        "%s classification report saved.",
        dataset_name
    )

    return report_path


# =============================================================================
# EXPLORATORY DATA ANALYSIS PIPELINE
# =============================================================================

def generate_dataset_visualizations(
    dataframe: pd.DataFrame,
    dataset_name: str,
    target_column: str
) -> None:
    """
    Generates all exploratory visualizations before
    model training.

    Parameters
    ----------
    dataframe : DataFrame

    dataset_name : str

    target_column : str
    """

    LOGGER.info(
        "%s | Generating exploratory figures...",
        dataset_name
    )

    plot_missing_values(
        dataframe,
        dataset_name
    )

    plot_correlation_heatmap(
        dataframe,
        dataset_name
    )

    plot_class_distribution(
        dataframe,
        target_column,
        dataset_name
    )

    plot_feature_distributions(
        dataframe,
        dataset_name
    )

    plot_boxplots(
        dataframe,
        dataset_name
    )

    LOGGER.info(
        "%s | Exploratory figures completed.",
        dataset_name
    )


# =============================================================================
# MODEL EVALUATION VISUALIZATION PIPELINE
# =============================================================================

def generate_evaluation_visualizations(
    dataset_name: str,
    model,
    feature_names,
    y_test,
    probabilities,
    confusion
) -> None:
    """
    Generates every evaluation figure after
    model prediction.
    """

    plot_confusion_matrix(
        confusion,
        dataset_name
    )

    plot_roc_curve(
        y_test,
        probabilities,
        dataset_name
    )

    plot_precision_recall_curve(
        y_test,
        probabilities,
        dataset_name
    )

    plot_feature_importance(
        model,
        feature_names,
        dataset_name
    )


# =============================================================================
# COMPLETE TRAINING PIPELINE
# =============================================================================

def train_single_dataset(
    dataset_name: str,
    dataset_config: dict
):
    """
    Executes the complete workflow for a single dataset.

    Workflow
    --------
    1. Load dataset
    2. Clean dataset
    3. Generate EDA figures
    4. Split data
    5. Fit preprocessing pipeline
    6. Train Random Forest
    7. Evaluate model
    8. Save reports
    9. Save model
    10. Save evaluation figures

    Returns
    -------
    dict
        Dictionary containing evaluation results.
    """

    print_title(
        f"TRAINING {dataset_name.upper()} MODEL"
    )

    (
        dataframe,
        preprocessing_pipeline,
        label_encoder,
        numerical_columns,
        categorical_columns
    ) = prepare_dataset(
        dataset_name,
        dataset_config
    )

    generate_dataset_visualizations(
        dataframe,
        dataset_name,
        dataset_config["Target"]
    )

    X, y = split_features_and_target(
        dataframe,
        dataset_config["Target"]
    )

    (
        X_train,
        X_test,
        y_train,
        y_test
    ) = perform_train_test_split(
        X,
        y
    )

    (
        X_train_processed,
        X_test_processed
    ) = preprocess_training_data(
        preprocessing_pipeline,
        X_train,
        X_test
    )

    model = create_random_forest_model()

    model = train_model(
        model,
        X_train_processed,
        y_train,
        dataset_name
    )

    (
        predictions,
        probabilities
    ) = generate_predictions(
        model,
        X_test_processed
    )

    metrics = evaluate_model(
        y_test,
        predictions
    )

    report = generate_classification_report(
        y_test,
        predictions
    )

    confusion = compute_confusion_matrix(
        y_test,
        predictions
    )

    feature_names = get_feature_names(
        preprocessing_pipeline,
        numerical_columns,
        categorical_columns
    )

    save_classification_report(
        report,
        dataset_name
    )

    save_model(
        model,
        preprocessing_pipeline,
        label_encoder,
        dataset_name
    )

    generate_evaluation_visualizations(
        dataset_name,
        model,
        feature_names,
        y_test,
        probabilities,
        confusion
    )

    LOGGER.info(
        "%s training pipeline completed successfully.",
        dataset_name
    )

    return {

        "Dataset": dataset_name,

        "Accuracy": metrics["Accuracy"],

        "Precision": metrics["Precision"],

        "Recall": metrics["Recall"],

        "F1Score": metrics["F1Score"]

    }


# =============================================================================
# SUMMARY TABLE
# =============================================================================

def display_training_summary(
    results: list
) -> None:
    """
    Displays a formatted summary of the trained models.

    Parameters
    ----------
    results : list
        List of dictionaries returned by
        train_single_dataset().
    """

    summary = pd.DataFrame(results)

    print_separator()

    print("TRAINING SUMMARY")

    print_separator()

    print(summary)

    print_separator()

    LOGGER.info(
        "Training summary generated."
    )

In [20]:
# =============================================================================
# MAIN TRAINING ORCHESTRATION
# =============================================================================

def main() -> None:
    """
    Main entry point for the training application.

    This function coordinates the complete training process for all
    configured datasets. For each disease dataset, it performs:

        1. Dataset loading
        2. Data cleaning and preprocessing
        3. Exploratory data analysis (EDA)
        4. Model training
        5. Model evaluation
        6. Report generation
        7. Model serialization
        8. Figure generation

    Finally, it produces a comparison chart of model accuracies and
    displays a summary table.

    Returns
    -------
    None
    """

    print_title(
        "MACHINE LEARNING-BASED EARLY RISK PREDICTION FOR COMMON DISEASES"
    )

    LOGGER.info("Starting training workflow...")

    training_results = []

    accuracy_results = {}

    for dataset_name, dataset_config in DATASETS.items():

        try:

            result = train_single_dataset(
                dataset_name,
                dataset_config
            )

            training_results.append(result)

            accuracy_results[
                dataset_name
            ] = result["Accuracy"]

        except Exception as error:

            LOGGER.exception(
                "Training failed for %s.",
                dataset_name
            )

            LOGGER.error(
                "Reason: %s",
                error
            )

            # Continue with the remaining datasets even if one fails.
            continue

    if len(training_results) == 0:

        LOGGER.error(
            "No models were successfully trained."
        )

        return

    # -------------------------------------------------------------------------
    # Generate cross-model comparison figure
    # -------------------------------------------------------------------------

    LOGGER.info(
        "Generating accuracy comparison figure..."
    )

    plot_accuracy_comparison(
        accuracy_results
    )

    # -------------------------------------------------------------------------
    # Display training summary
    # -------------------------------------------------------------------------

    display_training_summary(
        training_results
    )

    LOGGER.info(
        "Successfully trained %d model(s).",
        len(training_results)
    )

    LOGGER.info(
        "Training workflow completed successfully."
    )

    print_separator()

    print("PROJECT TRAINING COMPLETED SUCCESSFULLY")

    print_separator()

    print(f"Models saved to  : {MODEL_DIR}")

    print(f"Reports saved to : {REPORT_DIR}")

    print(f"Figures saved to : {FIGURE_DIR}")

    print_separator()


# =============================================================================
# SCRIPT ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    try:

        main()

    except KeyboardInterrupt:

        LOGGER.warning(
            "Training interrupted by user."
        )

        print("\nTraining cancelled.")

    except Exception as error:

        LOGGER.exception(
            "Unexpected fatal error occurred."
        )

        print_separator()

        print("TRAINING TERMINATED")

        print_separator()

        print(f"Error: {error}")

        print_separator()

2026-06-28 15:57:33,413 | INFO | Starting training workflow...
2026-06-28 15:57:33,415 | INFO | Loading Heart dataset...
2026-06-28 15:57:33,429 | INFO | Heart Dataset Shape: (1024, 15)
2026-06-28 15:57:33,431 | INFO | Heart dataset loaded successfully.
2026-06-28 15:57:33,445 | INFO | Heart | Removed 0 duplicate rows.
2026-06-28 15:57:33,452 | INFO | Heart | Removed 0 rows with critical missing values.
2026-06-28 15:57:33,494 | INFO | Detected 14 numerical features.
2026-06-28 15:57:33,497 | INFO | Detected 0 categorical features.
2026-06-28 15:57:33,499 | INFO | Heart preprocessing complete.
2026-06-28 15:57:33,502 | INFO | Heart | Generating exploratory figures...
2026-06-28 15:57:33,515 | INFO | Heart | No missing values detected.


MACHINE LEARNING-BASED EARLY RISK PREDICTION FOR COMMON DISEASES
TRAINING HEART MODEL


2026-06-28 15:57:36,784 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:57:36,810 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:58:06,363 | INFO | Heart | Exploratory figures completed.
2026-06-28 15:58:06,369 | INFO | Feature matrix shape: (1024, 14) | Target shape: (1024,)
2026-06-28 15:58:06,384 | INFO | Training samples : 819
2026-06-28 15:58:06,386 | INFO | Testing samples  : 205
2026-06-28 15:58:06,388 | INFO | Fitting preprocessing pipeline...
2026-06-28 15:58:06,436 | INFO | Preprocessing complete.
2026-06-28 15:58:06,438 | INFO | Processed training shape: (819, 14)
2026-06-28 15:58:06,441 | INFO | Processed testing shape: (205, 14)
2026-06-28 15:58:06

TRAINING DIABETES MODEL


2026-06-28 15:58:19,415 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:58:19,434 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:58:39,285 | INFO | Diabetes | Exploratory figures completed.
2026-06-28 15:58:39,290 | INFO | Feature matrix shape: (768, 8) | Target shape: (768,)
2026-06-28 15:58:39,300 | INFO | Training samples : 614
2026-06-28 15:58:39,303 | INFO | Testing samples  : 154
2026-06-28 15:58:39,305 | INFO | Fitting preprocessing pipeline...
2026-06-28 15:58:39,348 | INFO | Preprocessing complete.
2026-06-28 15:58:39,350 | INFO | Processed training shape: (614, 8)
2026-06-28 15:58:39,353 | INFO | Processed testing shape: (154, 8)
2026-06-28 15:58:39,3

TRAINING KIDNEY MODEL


2026-06-28 15:58:53,565 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:58:53,584 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:59:22,586 | INFO | Kidney | Exploratory figures completed.
2026-06-28 15:59:22,591 | INFO | Feature matrix shape: (400, 13) | Target shape: (400,)
2026-06-28 15:59:22,605 | INFO | Training samples : 320
2026-06-28 15:59:22,607 | INFO | Testing samples  : 80
2026-06-28 15:59:22,610 | INFO | Fitting preprocessing pipeline...
2026-06-28 15:59:22,654 | INFO | Preprocessing complete.
2026-06-28 15:59:22,656 | INFO | Processed training shape: (320, 13)
2026-06-28 15:59:22,658 | INFO | Processed testing shape: (80, 13)
2026-06-28 15:59:22,66

TRAINING STROKE MODEL


2026-06-28 15:59:34,183 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:59:34,207 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-28 15:59:49,932 | INFO | Stroke | Exploratory figures completed.
2026-06-28 15:59:49,938 | INFO | Feature matrix shape: (5110, 11) | Target shape: (5110,)
2026-06-28 15:59:49,954 | INFO | Training samples : 4088
2026-06-28 15:59:49,956 | INFO | Testing samples  : 1022
2026-06-28 15:59:49,959 | INFO | Fitting preprocessing pipeline...
2026-06-28 15:59:50,053 | INFO | Preprocessing complete.
2026-06-28 15:59:50,056 | INFO | Processed training shape: (4088, 22)
2026-06-28 15:59:50,057 | INFO | Processed testing shape: (1022, 22)
2026-06-28 15:

TRAINING SUMMARY
    Dataset  Accuracy  Precision    Recall   F1Score
0     Heart  1.000000   1.000000  1.000000  1.000000
1  Diabetes  0.740260   0.659091  0.537037  0.591837
2    Kidney  1.000000   1.000000  1.000000  1.000000
3    Stroke  0.951076   0.500000  0.020000  0.038462
PROJECT TRAINING COMPLETED SUCCESSFULLY
Models saved to  : C:\Users\FidelisNT\Documents\Risk Prediction For Common Diseases\backend\app\models
Reports saved to : C:\Users\FidelisNT\Documents\Risk Prediction For Common Diseases\reports
Figures saved to : C:\Users\FidelisNT\Documents\Risk Prediction For Common Diseases\figures
